# 📈 ขั้นตอนที่ 5: Evaluation & Value Betting
### (การประเมินผลโมเดลและการนำไปใช้งาน)
---
**วัตถุประสงค์ในขั้นตอนนี้:**
1. ประเมินผลโมเดลบนชุดข้อมูลทดสอบ (Test Set) ด้วยค่า Accuracy, Brier Score และ Log Loss
2. เปรียบเทียบประสิทธิภาพของโมเดลกับ Baseline ของตลาดโต๊ะบอล (No-Vig Odds)
3. จำลองผลกำไรจากการลงทุน (Backtest Simulation & ROI) เมื่อเลือกแทงเฉพาะนัดที่มี Edge $\ge$ 6%
4. พยากรณ์คู่บอลล่วงหน้าและแสดงผลเป็นการ์ดสรุป Match Cards

## 1. นำเข้าไลบรารีและโหลดโมเดล

In [ ]:
import os, sys
# จัดการ Working Directory ให้ชี้ไปที่โฟลเดอร์หลักของโปรเจกต์เสมอ
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, brier_score_loss

# โหลดชุดข้อมูลและโมเดล
df = pd.read_csv('data/preprocessed_matches.csv')
model_data = joblib.load('models/student_model.pkl')
model = model_data['model']
feature_cols = model_data['feature_columns']

split_idx = int(len(df) * 0.8)
df_test = df.iloc[split_idx:].copy()
X_test = df_test[feature_cols]
y_test = df_test[['ah_label', 'ou_label']]

print(f"โหลดข้อมูลชุดทดสอบ: {len(X_test)} นัด")

## 2. ประเมินความแม่นยำทางสถิติ (Statistical Evaluation)

In [ ]:
# พยากรณ์ความน่าจะเป็น
probs = model.predict_proba(X_test)
p_fav_model = probs[0][:, 1]
p_over_model = probs[1][:, 1]

acc_ah = accuracy_score(y_test['ah_label'], p_fav_model >= 0.5)
acc_ou = accuracy_score(y_test['ou_label'], p_over_model >= 0.5)
brier_ah = brier_score_loss(y_test['ah_label'], p_fav_model)
brier_ou = brier_score_loss(y_test['ou_label'], p_over_model)

print("=== ผลการประเมินโมเดลบน Test Set ===")
print(f"Asian Handicap Accuracy: {acc_ah:.2%} | Brier Score: {brier_ah:.4f}")
print(f"Over/Under Accuracy:     {acc_ou:.2%} | Brier Score: {brier_ou:.4f}")

## 3. การจำลองผลตอบแทนทางการเงิน (Value Betting Simulation)
คำนวณ **Edge** = ความน่าจะเป็นของโมเดล - ราคาแฟร์ของโต๊ะ
ลงทุนเฉพาะคู่ที่ $\text{Edge} \ge 6\%$ (เดิมพันคงที่ 1 หน่วยต่อนัด)

In [ ]:
df_test['p_fav_model'] = p_fav_model
df_test['p_over_model'] = p_over_model

# คำนวณราคาแฟร์ของโต๊ะ (No-Vig)
fav_inv = 1.0 / df_test['ah_fav_price']
dog_inv = 1.0 / df_test['ah_dog_price']
df_test['p_fav_market'] = fav_inv / (fav_inv + dog_inv)
df_test['p_dog_market'] = dog_inv / (fav_inv + dog_inv)

# หา Edge สำหรับฝั่งทีมรอง (AH_dog)
df_test['dog_edge'] = (1.0 - df_test['p_fav_model']) - df_test['p_dog_market']

# คัดเฉพาะนัดที่ Edge >= 6%
bets = df_test[df_test['dog_edge'] >= 0.06].copy()
bets['profit'] = np.where(bets['ah_label'] == 0, bets['ah_dog_price'] - 1.0, -1.0)

total_bets = len(bets)
total_profit = bets['profit'].sum()
roi = (total_profit / total_bets) * 100 if total_bets > 0 else 0.0
win_rate = (bets['profit'] > 0).mean() * 100 if total_bets > 0 else 0.0

print("=== ผลการจำลองการลงทุนจริง (Edge >= 6%) ===")
print(f"จำนวนนัดที่เข้าเกณฑ์ลงทุน: {total_bets} นัด")
print(f"อัตราการชนะ (Win Rate):    {win_rate:.2f}%")
print(f"กำไรสุทธิสะสม (Profit):     +{total_profit:.2f} หน่วย")
print(f"ผลตอบแทนต่อการลงทุน (ROI):  +{roi:.2f}%")

## 4. แสดงผลการทำนายคู่แข่งขันล่วงหน้า (Live Match Cards)

In [ ]:
from IPython.display import Markdown, display
import os

report_file = 'reports/upcoming_picks.md'
if os.path.exists(report_file):
    with open(report_file, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content))
else:
    print("ยังไม่มีไฟล์รายงาน upcoming_picks.md ให้รัน predict_upcoming.py ก่อน")